<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Run meeting LLM — 2026 Gemma 4 Good Hackathon

> One notebook. Real municipal records (PDFs + audio). Four Gemma 4 capabilities, side by side, on the same files judges can open themselves.

## What this notebook proves

| # | Gemma 4 capability | What you see |
|---|---|---|
| 1 | **Native multimodal PDF parsing** | Reads scanned minutes with **zero digital text** as images — surfaces "dark data" that `pdftotext` can't touch. |
| 2 | **Adjustable visual token budget** | Per-page routing: financial tables / scans → HIGH (~1,120 tok), text-heavy minutes → LOW (~64 tok). Dollar-amount fidelity without paying scan rates on prose. |
| 3 | **Built-in thinking mode** | Runs `prompts/policy_analysis_v1.md` with `include_thoughts=True` — judges see the model's *reasoning trace*, not just the verdict (e.g. April 2026 Tuscaloosa nuisance demolitions: public safety vs. property rights). |
| 4 | **Long-context drift detection** | 15-minute audio chunks → per-chunk policy analysis → **Policy Drift Detector** pass that tracks how a subject (e.g. Tuscaloosa downtown zoning) shifts from origination through final amendment. |

**Sample data** ships pre-staged on Drive: Tuscaloosa AL (county + city) and Big Timber MT (Sweet Grass County + city) — Courthouse Project rankings, airport contract amendments, routine minutes, and full meeting audio.

## Which Gemma 4 variant — and where does Gemini fit?

This notebook supports **Hugging Face** local Gemma 4 (`GOVERNANCE_LLM_BACKEND=huggingface`, default) or Google's **`google-genai`** API. You can pick **different models for triage vs. deep analysis** to control cost.

### Gemma 4 variants (released April 2026, open weights)

Full sizing / benchmark / license details: **[Gemma 4 model card](https://ai.google.dev/gemma/docs/core/model_card_4)**.

> **Heads up — id visibility varies per API project.** The Gemma 4 product page lists four sizes (E2B, E4B, 26B A4B, 31B). All four are released as open weights, but the **ids your AI Studio key sees via `client.models.list()` depend on your project tier and which previews you've opted into**. On many projects only `gemma-4-26b-a4b-it` and `gemma-4-31b-it` are returned today; requesting `gemma-4-e4b-it` against such a key returns 404 NOT_FOUND. **§3 API key** prints the full `models.list()` *before* any fallback, then resolves ids — run that cell before **§4 Gatekeeper** so you can pick a real `GOVERNANCE_GATEKEEPER_MODEL`.


| Variant | Size | Typically visible to `models.list()`? | Best for | Notes |
|---|---|---|---|---|
| [**E2B**](https://ai.google.dev/gemma/docs/core/model_card_4#e2b) (`gemma-4-e2b-it`) | Effective 2B | ⚠️ depends on project | Lightweight triage on mobile/laptop/edge | Multimodal (text + image + video) + **native audio input**. Open weights only |
| [**E4B**](https://ai.google.dev/gemma/docs/core/model_card_4#e4b) (`gemma-4-e4b-it`) | Effective 4B | ⚠️ depends on project | Cheap triage with better accuracy than E2B | Same modality support as E2B. Open weights only |
| [**26B MoE**](https://ai.google.dev/gemma/docs/core/model_card_4#26b-moe) (`gemma-4-26b-a4b-it`, default) | 26B mixture-of-experts | ✅ commonly | Heavy multimodal analysis — also doubles as triage since only  4B params are active per token | Cheapest Gemma 4 currently served |
| [**31B Dense**](https://ai.google.dev/gemma/docs/core/model_card_4#31b-dense) (`gemma-4-31b-it`) | 31B dense | ✅ commonly | Deepest reasoning, autonomous workflows, function calling, structured JSON | SOTA Gemma 4 — pair with `GOVERNANCE_FORCE_THINKING=1` if your AI Studio project enables thinking mode |

### Gemini 2.5+ ([`gemini-2.5-flash`](https://ai.google.dev/gemini-api/docs/models#gemini-2.5-flash), [`gemini-2.5-pro`](https://ai.google.dev/gemini-api/docs/models#gemini-2.5-pro))

Proprietary, AI-Studio-only. Native support for **`thinking_config`** (`thinking_budget`, `include_thoughts`) — that's the feature Demo 3 ("reasoning trace") really wants.

### Recommended setup for this notebook

Set **two** env vars and let the cells route work to the right model:

```python
os.environ["GOVERNANCE_GATEKEEPER_MODEL"] = "gemma-4-26b-a4b-it"  # triage (cheapest Gemma 4 currently hosted)
os.environ["GOVERNANCE_GENAI_MODEL"]      = "gemma-4-26b-a4b-it"  # heavy demos (swap to gemma-4-31b-it for deeper reasoning)
# Optional: for Demo 3 reasoning trace specifically →
# os.environ["GOVERNANCE_GENAI_MODEL"] = "gemini-2.5-flash"
```

If/when AI Studio starts serving the E2B / E4B endpoints, point `GOVERNANCE_GATEKEEPER_MODEL` at one of those for a real cost win on triage.

| Cell | Uses | Why |
|---|---|---|
| Cell 11 — Gatekeeper triage | `GATEKEEPER_MODEL` | One short prompt per file just to decide *is this even a meeting record?* → cheapest model wins |
| Cells 14, 16, 19, 21, 23 — Demos 1–5 | `GENAI_MODEL` | Full multimodal OCR, page-budget routing, reasoning trace, long-audio drift detection → needs the larger model |

### Feature support matrix

| Feature | Gemini 2.5+ | Gemma 4 (all variants) |
|---|---|---|
| Native multimodal (PDF / image / audio in one request) | ✅ | ✅ |
| `media_resolution` HIGH/LOW per image | ✅ | ✅ |
| `response_mime_type=application/json` + `response_schema` | ✅ | ✅ |
| `thinking_config` (`thinking_budget`, `include_thoughts`) | ✅ | ⚠️ The 26B MoE alias returns `400: Thinking budget is not supported for this model` — the helper auto-skips `thinking_config` for `gemma-*` ids. Set `GOVERNANCE_FORCE_THINKING=1` if your Gemma 4 variant actually supports it via the SDK config. |

**What this means for Demo 3:** with the default Gemma 26B MoE, the helper runs the deconstruction prompt as a normal generation (JSON output works; the `.thoughts.md` reasoning-trace file will be empty). Switch `GOVERNANCE_GENAI_MODEL` to `gemini-2.5-flash` (or `gemma-4-31b-it` + `GOVERNANCE_FORCE_THINKING=1`) to capture real reasoning traces.

### Beyond Gemma 4 — specialized variants worth knowing

Google publishes a family of task-specific Gemma models. Two would meaningfully extend this pipeline; the rest don't fit a governance-meeting use case. Both are wired in below as **Demo 6** (EmbeddingGemma clustering) and a **Safety Review** pass (ShieldGemma).

| Variant | Where it would slot into this pipeline | Why it's a fit |
|---|---|---|
| [**EmbeddingGemma**](https://ai.google.dev/gemma/docs/embeddinggemma/model_card) | A "Demo 6" pass over the JSON outputs in `03_processed_outputs/02_gemma_json/` | Embeds each policy decision / agenda item; lets you cluster similar issues across jurisdictions (e.g. all "downtown zoning" decisions in MT and AL), dedupe repeat agenda items across months, and power a semantic search over the entire extracted corpus |
| [**ShieldGemma**](https://ai.google.dev/gemma/docs/shieldgemma/model_card) | A safety-review pass after Demo 5 (contact image enrichment) and after Demo 4's audio chunks | Demo 5 returns *perceived* demographics on public officials, and meeting audio can include public commentary on sensitive topics. ShieldGemma flags policy-violating content before any of this is published downstream — a real civic-tech provenance win |


## Before you Run All

1. **Get a Hugging Face token.** [Settings → Access Tokens](https://huggingface.co/settings/tokens) → *New token* (read access is enough) → accept the [Gemma 4 E4B license](https://huggingface.co/google/gemma-4-E4B-it).
2. **Add it as a Colab Secret.** Left sidebar 🔑 → *Add new secret* → name it exactly `HF_TOKEN`, paste the value, toggle **Notebook access ON**.
   For **Google AI Studio** (`GOVERNANCE_LLM_BACKEND=google`), use a separate secret `GEMINI_API_KEY`.
3. **Mount your Google Drive.** The next cell calls `drive.mount("/content/drive")` for you — no manual code needed. When you hit *Run All*:
   - A **"Permit this notebook to access your Google Drive files?"** dialog appears → click **Connect to Google Drive**.
   - A second window pops up asking you to **sign in**. Pick the Google account that owns the `CommunityOne/` folder (or has Shared-Drive access to it). Approve the requested scopes.
   - The mount appears at `/content/drive/MyDrive/...` and the bootstrap cell prints `Mounted at /content/drive`.

   **Expected Drive layout** (the resolver probes both; first existing folder wins):
   ```
   MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good/   ← preferred (hackathon dataset)
     ├── 01_raw_inputs/<STATE>/<county|municipality>/<jurisdiction>/…
     ├── 02_reference_data/
     └── 03_processed_outputs/                          ← created on first run

   MyDrive/CommunityOne/governance_pipeline_data/       ← fallback (canonical layout)
     └── …same children…
   ```
   The pre-staged Tuscaloosa + Big Timber sample lives at the first path — judges just approve the OAuth prompt and the resolver finds it. If neither folder exists, the bootstrap cell will print a ✓/· table of every path it probed plus a fix command.

   **Using a Shared Drive instead?** No config needed — paths matching `Shareddrives/*/CommunityOne/hackathons/2026_Gemma_4_Good` and `Shareddrives/*/CommunityOne/governance_pipeline_data` are probed automatically.

   **Custom location?** Set `os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"] = "/content/drive/MyDrive/your/path"` in a cell *before* the bootstrap, or point `GOVERNANCE_RAW_INPUTS_ROOT` at just the `01_raw_inputs/` folder if it lives outside the pipeline root.
4. **(Optional) Override the model id.** Default is the Gemma 4 alias `gemma-4-26b-a4b-it`. If AI Studio lists a different id for your project, set `GOVERNANCE_GENAI_MODEL` before running cell 8.
5. **Run All.** Cells 1–6 bootstrap the env (Drive mount, repo sync, secrets, caps). Cells 7–25 execute the four demos and write JSON + human-readable summaries back to `03_processed_outputs/` on Drive.

> **Cost guardrails:** every demo honors `GOVERNANCE_DEMO_MAX_*` env vars (see cell 8). Defaults are sized for a free-tier judging run — widen them only after a dry pass.

## 1) Bootstrap repo + Drive

On Colab the next cell mounts Google Drive, clones (or refreshes) `open-navigator`, and resolves the pipeline data root. The resolver probes — in order — `GOVERNANCE_PIPELINE_DATA_ROOT`, then `MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good`, then `MyDrive/CommunityOne/governance_pipeline_data`, then matching Shared-Drive paths. First existing folder wins; if none match you'll get a diagnostic listing every candidate so you can fix the layout. `OPEN_NAVIGATOR_ROOT` overrides repo discovery.

In [18]:
# Find the open-navigator repo, refresh it from GitHub, then import the resolver.
# Order matters: any cached scripts.colab.* modules must be evicted BEFORE the
# import below, or git-reset --hard won't take effect this session.
import os
import shutil
import sys
from pathlib import Path

REPO_PATH = Path("/content/open-navigator")
EPHEMERAL_DATA_DIR = Path("/content/governance_pipeline_data")

# 1) Clone if missing.
if not REPO_PATH.is_dir():
    print(f"Cloning open-navigator into {REPO_PATH}...")
    rc = os.system(f"git clone https://github.com/getcommunityone/open-navigator.git {REPO_PATH}")
    if rc != 0:
        raise RuntimeError(f"git clone failed (exit {rc})")

os.environ.setdefault("OPEN_NAVIGATOR_ROOT", str(REPO_PATH))

# 2) git fetch + reset --hard origin/main BEFORE the first import, so Python loads
#    the latest colab_paths.py from disk (not a cached stale version).
RUN_GIT_UPDATE = True
if RUN_GIT_UPDATE and (REPO_PATH / ".git").is_dir():
    print("🔄 Fetching latest open-navigator from GitHub...")
    rc = os.system(
        f"cd {REPO_PATH} && git config core.hooksPath /dev/null "
        "&& git fetch origin && git reset --hard origin/main"
    )
    if rc != 0:
        print(f"⚠️  git update returned exit {rc} — continuing with whatever is on disk.")

# 3) Defensive cleanup of stale session state from prior runs.
#    Without this, re-running the cell silently uses cached old code / paths.
for _mod in [m for m in list(sys.modules) if m.startswith("scripts.colab") or m.startswith("scripts.utils.gdrive_paths")]:
    sys.modules.pop(_mod, None)
_stale_root = os.environ.pop("GOVERNANCE_PIPELINE_DATA_ROOT", None)
if _stale_root:
    print(f"   Cleared stale GOVERNANCE_PIPELINE_DATA_ROOT={_stale_root}")

# 4) Remove the ephemeral /content/governance_pipeline_data shell if a previous
#    notebook version (or PIPE.ensure_dirs) left it behind. Real data lives on
#    mounted Drive — this folder is on disposable runtime disk and confuses judges.
if EPHEMERAL_DATA_DIR.is_dir():
    print(f"   Removing stale ephemeral folder {EPHEMERAL_DATA_DIR} "
          "(real data lives on Drive — this is just an empty shell from a prior bug).")
    shutil.rmtree(EPHEMERAL_DATA_DIR, ignore_errors=True)

# 5) Put the repo on sys.path so `from scripts.colab.colab_paths import …` works.
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# 6) Now the first import sees the freshly-pulled code.
from scripts.colab.colab_paths import maybe_mount_google_drive, setup_notebook_paths

maybe_mount_google_drive()

try:
    PATHS = setup_notebook_paths()
except RuntimeError as e:
    # Resolver couldn't find the data root on Drive. Surface its diagnostic +
    # show what IS visible under the mount so the user can spot the typo.
    print("\n❌ setup_notebook_paths() failed:\n")
    print(str(e))
    print("\nCurrent /content/drive view:")
    for p in (Path("/content/drive"), Path("/content/drive/MyDrive"),
              Path("/content/drive/MyDrive/CommunityOne"),
              Path("/content/drive/MyDrive/CommunityOne/hackathons")):
        if p.is_dir():
            children = sorted([c.name for c in p.iterdir()])[:20]
            print(f"  {p}: {children}")
        else:
            print(f"  {p}: (missing)")
    raise

print("Runtime:", "Google Colab" if PATHS.in_colab else "Local")
print("Repo:                 ", PATHS.project_path)
print("Governance data root: ", PATHS.governance_pipeline_data)


🔄 Fetching latest open-navigator from GitHub...
   Cleared stale GOVERNANCE_PIPELINE_DATA_ROOT=/content/drive/MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Google Colab
Repo:                  /content/open-navigator
Governance data root:  /content/drive/MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good


In [19]:
import os
import sys

# Repo path is fixed in cell 4 (git fetch + reset --hard ran there).
proj = PATHS.project_path

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

# Pin the resolved Drive path so downstream modules read it from the env.
os.environ.setdefault("GOVERNANCE_PIPELINE_DATA_ROOT", str(PATHS.governance_pipeline_data))

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
# Create only the subfolders we will write into (03_processed_outputs/*). We do
# NOT create 01_raw_inputs here — if the resolver pointed somewhere wrong, an
# empty 01_raw_inputs would mask the bug.
for _sub in (PIPE.transcripts, PIPE.gemma_json, PIPE.human_summaries):
    _sub.mkdir(parents=True, exist_ok=True)

from governance_meeting_llm import (
    AUDIO_EXTS,
    PDF_EXTS,
    TOKEN_BUDGET_HIGH,
    TOKEN_BUDGET_LOW,
    call_google_genai_multimodal,
    transcribe_audio_with_gemma,
    TRANSCRIPTION_SUPPORTED_LANGUAGES,
    chunk_audio_ffmpeg,
    classify_pdf_page_heuristic,
    extract_pdf_digital_text,
    is_scanned_pdf,
    load_contacts_lookup,
    load_meeting_data_lookup,
    load_text_file,
    merge_contacts_by_jurisdiction,
    merge_meeting_data_by_jurisdiction,
    mime_for,
    mirror_output_path,
    parse_policy_analysis_response,
    policy_drift_summarize,
    render_pdf_pages,
    walk_raw_inputs,
    embed_text_with_gemma,
    cosine_similarity_matrix,
    shield_review_text,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)


Loaded prompt chars: 39836
Pipeline data root: /content/drive/MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good


In [20]:
import contextlib
import io
import sys
from pathlib import Path
from datetime import datetime, timezone

def execute_and_log_verbose(
    func_to_execute,
    step_name: str,
    output_threshold: int = 5,
    log_always: bool = False,
    log_base_dir: Path = None,
    **kwargs
):
    def _create_log_dir_and_path(s_name: str, b_dir: Path):
        log_dir = b_dir / "_notebook_logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        return log_dir / f"{s_name}_console_log_{stamp}.txt"

    if log_base_dir is None:
        raise ValueError("log_base_dir must be provided.")

    log_path = _create_log_dir_and_path(step_name, log_base_dir)
    buffer = io.StringIO()
    return_value = None

    print(f"Executing '{step_name}'. Console output redirected to '{log_path.relative_to(log_base_dir.parent)}' if verbose (>{output_threshold} lines) or if log_always=True.")

    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = io.TextIOWrapper(buffer, encoding=sys.stdout.encoding, line_buffering=True)
    sys.stderr = io.TextIOWrapper(buffer, encoding=sys.stderr.encoding, line_buffering=True)

    try:
        return_value = func_to_execute(**kwargs)
    except Exception as e:
        # Capture tracebacks in the buffer as well
        import traceback
        traceback.print_exc(file=sys.stderr)
        # Re-raise after logging the buffer content
        with open(log_path, 'w', encoding='utf-8') as f:
            f.write(buffer.getvalue())
        print(f"An error occurred during '{step_name}'. Full log: '{log_path.relative_to(log_base_dir.parent)}'", file=old_stderr)
        raise
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr

    captured_output = buffer.getvalue()
    lines = captured_output.strip().split('\n') if captured_output.strip() else []

    if log_always or len(lines) > output_threshold:
        with open(log_path, 'w', encoding='utf-8') as f:
            f.write(captured_output)
        if not log_always: # only print summary if we redirected
            print(f"  Console output for '{step_name}' exceeded {output_threshold} lines and was redirected to '{log_path.relative_to(log_base_dir.parent)}'.")
    else:
        # If not verbose, print to console, but avoid duplicating the "Executing..." line
        if captured_output:
            # Only print the content that wasn't already printed by the initial 'Executing...' line
            initial_message_line = f"Executing '{step_name}'. Console output redirected to '{log_path.relative_to(log_base_dir.parent)}' if verbose (>{output_threshold} lines) or if log_always=True."
            # Check if the captured output actually contains the initial message line, before stripping it.
            # This avoids issues if the initial message itself was partially truncated or altered by Python's buffering.
            if captured_output.startswith(initial_message_line.split('\n')[0]): # Check against first line of initial message
                captured_output = captured_output[len(initial_message_line):].lstrip('\n')

            # Now print the actual captured output to console
            print(captured_output, end='') # preserve newlines etc.

    return return_value


## 2) Install dependencies

Colab already ships **torch**, **torchvision**, **torchaudio**, **accelerate**, and **pillow** — do not
`pip install -U` them (especially **transformers** with `-U`, which can bump torch to 2.12 while
torchaudio stays on 2.10 and triggers resolver errors). The next cell installs only missing
packages **without** `-U`. If you already upgraded torch, use **Runtime → Restart session** and
re-run from §2.

`google-genai` is the Gemini / Gemma 4 client. `pymupdf` renders PDF pages to
images and probes their digital-text layer for the per-page token-budget demo.
`pdf2image` (used by the Gatekeeper step 0) needs the system package
`poppler-utils` — pre-installed on Colab; locally `apt-get install -y poppler-utils`
on Debian/Ubuntu or `brew install poppler` on macOS. `ffmpeg` ships with Colab;
locally `apt-get install -y ffmpeg` if you plan to run **Demo 4** (audio chunking)
or the Gatekeeper audio gate.

In [21]:
# poppler-utils is required by pdf2image (Gatekeeper step 0). Skipped silently off Colab.
import shutil, subprocess
if not shutil.which("pdftoppm"):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
    except FileNotFoundError:
        pass

# Only install what Colab lacks. No -U on transformers (avoids upgrading torch vs torchaudio).
%pip install -q "google-genai>=1.0" "transformers>=4.50" librosa pymupdf pdf2image

## 3) Backend + model + demo caps

**Hugging Face (default for E4B):** set `GOVERNANCE_LLM_BACKEND=huggingface` and
`GOVERNANCE_HF_MODEL_ID=google/gemma-4-E4B-it` (loads `AutoProcessor` +
`AutoModelForImageTextToText`; audio triage uses `AutoModelForMultimodalLM`).
Colab Secret **`HF_TOKEN`** (see *Before you Run All*). Accept the model license on Hugging Face.

**Google AI Studio:** `GOVERNANCE_LLM_BACKEND=google` + Colab Secret `GEMINI_API_KEY`.

Run this cell before **§4 Gatekeeper**.

The four demo cells honor a few env knobs so live runs don't blow through tokens:

| Env var | Default | What it caps |
|---|---|---|
| `GOVERNANCE_DEMO_MAX_PDFS_PER_JUR` | `3` | PDFs processed per jurisdiction in Demo 1 + 2 |
| `GOVERNANCE_DEMO_MAX_PAGES_PER_PDF` | `8` | Pages processed per PDF in Demo 2 |
| `GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR` | `1` | Audio files processed per jurisdiction in Demo 4 |
| `GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS` | `4` | 15-minute audio chunks processed per file in Demo 4 |
| `GOVERNANCE_DEMO_THINKING_BUDGET` | `-1` | Thinking-token budget for Demo 3 (`-1` = unlimited) |

In [22]:
import os

# Set the LLM backend to Google AI Studio
os.environ["GOVERNANCE_LLM_BACKEND"] = "google"

# Set models to use with Google AI Studio (e.g., gemma-4-26b-a4b-it or gemma-4-31b-it)
os.environ["GOVERNANCE_GATEKEEPER_MODEL"] = "gemma-4-26b-a4b-it"
os.environ["GOVERNANCE_GENAI_MODEL"] = "gemma-4-26b-a4b-it"

# Control logging of the full LLM model catalog. Set to "1" to enable.
os.environ["GOVERNANCE_LOG_LLM_CATALOG"] = "0"

print("GOVERNANCE_LLM_BACKEND set to 'google'.")

GOVERNANCE_LLM_BACKEND set to 'google'.


In [23]:
import os

# Backend: huggingface (local E4B) vs google (AI Studio API)
LLM_BACKEND = os.environ.get("GOVERNANCE_LLM_BACKEND", "huggingface").strip().lower()
USE_HF = LLM_BACKEND in ("huggingface", "hf", "local")
HF_MODEL_ID = os.environ.get("GOVERNANCE_HF_MODEL_ID", "google/gemma-4-E4B-it").strip()

try:
    from google.colab import userdata

    _get_secret = userdata.get
except ImportError:

    def _get_secret(name: str):
        return os.environ.get(name)


def _sanitize_key(value):
    """Strip whitespace + interior CR/LF so an accidental newline in a Colab
    Secret can't poison the HTTP Authorization header (httplib raises
    'Illegal header value' on any CR/LF in a header).
    """
    if not value:
        return value
    cleaned = (
        value.strip()
        .replace("\r", "")
        .replace("\n", "")
        .replace("\t", "")
        .replace(" ", "")
    )
    if cleaned != value.strip():
        print(
            f"   ⚠️  API key had {len(value) - len(cleaned)} stray whitespace/newline "
            "char(s) — stripped. Re-copy your token into the Colab Secret to silence this."
        )
    return cleaned


HF_TOKEN = _sanitize_key(_get_secret("HF_TOKEN"))
GEMINI_API_KEY = _sanitize_key(_get_secret("GEMINI_API_KEY") or _get_secret("GOOGLE_API_KEY"))

if USE_HF:
    if not HF_TOKEN:
        raise RuntimeError(
            "Set Colab Secret HF_TOKEN (or export HF_TOKEN in the environment). "
            "Create a token at https://huggingface.co/settings/tokens and accept "
            "the model license at https://huggingface.co/google/gemma-4-E4B-it"
        )
    os.environ["HF_TOKEN"] = HF_TOKEN
    API_KEY = HF_TOKEN
    print(f"LLM backend: Hugging Face | repo={HF_MODEL_ID}")
    if not GEMINI_API_KEY:
        print("   ℹ️  No GEMINI_API_KEY — Demo 6 (embeddings) and Shield need AI Studio or skip those cells.")
else:
    if not GEMINI_API_KEY:
        raise RuntimeError(
            "Set GEMINI_API_KEY (Colab Secret or environment). "
            "Create a key at https://aistudio.google.com/apikey"
        )
    if not GEMINI_API_KEY.startswith("AIza") or len(GEMINI_API_KEY) < 35:
        raise RuntimeError(
            f"GEMINI_API_KEY looks malformed (len={len(GEMINI_API_KEY)}, "
            f"prefix={GEMINI_API_KEY[:4]!r}). Google AI Studio keys start with 'AIza'."
        )
    API_KEY = GEMINI_API_KEY

_default_genai = HF_MODEL_ID if USE_HF else "gemma-4-26b-a4b-it"
_default_gate = HF_MODEL_ID if USE_HF else "gemma-4-26b-a4b-it"
GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", _default_genai).strip()
GATEKEEPER_MODEL = os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", _default_gate).strip() or GENAI_MODEL
# EmbeddingGemma for Demo 6 (cross-jurisdiction clustering).
EMBEDDING_MODEL = os.environ.get("GOVERNANCE_EMBEDDING_MODEL", "embeddinggemma-300m").strip()
# ShieldGemma for the safety-review pass.
SHIELD_MODEL = os.environ.get("GOVERNANCE_SHIELD_MODEL", "shieldgemma-9b").strip() or GENAI_MODEL

os.environ["GOVERNANCE_LLM_BACKEND"] = "huggingface" if USE_HF else "google"

_log_llm_catalog = os.environ.get("GOVERNANCE_LOG_LLM_CATALOG", "0") == "1"

if USE_HF:
    from gemma_hf_backend import (
        preload_gemma_hf,
        print_hf_model_catalog,
        resolve_hf_model_id,
    )

    if _log_llm_catalog:
        print_hf_model_catalog(
            requested=(GENAI_MODEL, GATEKEEPER_MODEL),
            role="Hugging Face (before load)",
        )
    GENAI_MODEL = resolve_hf_model_id(GENAI_MODEL)
    GATEKEEPER_MODEL = resolve_hf_model_id(GATEKEEPER_MODEL)
    print("Loading weights (ImageTextToText + MultimodalLM for audio)…")
    preload_gemma_hf(GENAI_MODEL, load_audio_variant=True)
    _resolver_client = None
    SHIELD_MODEL = resolve_hf_model_id(SHIELD_MODEL) if SHIELD_MODEL else GENAI_MODEL
else:
    from gatekeeper_triage import (
        resolve_model_id,
        print_available_models,
        _build_genai_client,
        _GEMMA_TRIAGE_FALLBACKS,
        _GEMMA_HEAVY_FALLBACKS,
    )
    _resolver_client = _build_genai_client(GEMINI_API_KEY)
    if _log_llm_catalog:
        print_available_models(
            _resolver_client,
            requested=(GENAI_MODEL, GATEKEEPER_MODEL, EMBEDDING_MODEL, SHIELD_MODEL),
            role="this API key",
        )
    GENAI_MODEL      = resolve_model_id(_resolver_client, GENAI_MODEL,      fallbacks=_GEMMA_HEAVY_FALLBACKS,  role="Heavy GenAI model")
    GATEKEEPER_MODEL = resolve_model_id(_resolver_client, GATEKEEPER_MODEL, fallbacks=_GEMMA_TRIAGE_FALLBACKS, role="Gatekeeper triage model")
    EMBEDDING_MODEL  = resolve_model_id(
        _resolver_client, EMBEDDING_MODEL,
        fallbacks=("embeddinggemma-300m", "text-embedding-004", "gemini-embedding-001"),
        role="Embedding model",
    )
    SHIELD_MODEL     = resolve_model_id(
        _resolver_client, SHIELD_MODEL,
        fallbacks=("shieldgemma-9b", "shieldgemma-2b", *_GEMMA_TRIAGE_FALLBACKS),
        role="Shield model",
    )

# Demo caps (override via env to widen / narrow scope).
MAX_PDFS_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_PDFS_PER_JUR", "3"))
MAX_PAGES_PER_PDF = int(os.environ.get("GOVERNANCE_DEMO_MAX_PAGES_PER_PDF", "8"))
MAX_AUDIO_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR", "1"))
MAX_AUDIO_CHUNKS = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS", "4"))
THINKING_BUDGET = int(os.environ.get("GOVERNANCE_DEMO_THINKING_BUDGET", "-1"))
DRIFT_FOCUS = os.environ.get("GOVERNANCE_DRIFT_FOCUS", "").strip() or None

print("LLM backend:", LLM_BACKEND)
print("GenAI model (demos):    ", GENAI_MODEL)
print("Gatekeeper model (triage):", GATEKEEPER_MODEL)
print("Embedding model (demo 6):  ", EMBEDDING_MODEL)
print("Shield model (safety pass):", SHIELD_MODEL)
print(
    "Caps:",
    f"pdfs/jur={MAX_PDFS_PER_JUR}",
    f"pages/pdf={MAX_PAGES_PER_PDF}",
    f"audio/jur={MAX_AUDIO_PER_JUR}",
    f"chunks/audio={MAX_AUDIO_CHUNKS}",
    f"thinking_budget={THINKING_BUDGET}",
)


── Embedding model: 'embeddinggemma-300m' not in models.list() — Gemma ids on this key ──
  gemma-4-26b-a4b-it
  gemma-4-31b-it
  Fallback order: ['embeddinggemma-300m', 'text-embedding-004', 'gemini-embedding-001']


── Shield model: 'shieldgemma-9b' not in models.list() — Gemma ids on this key ──
  gemma-4-26b-a4b-it
  gemma-4-31b-it
  Fallback order: ['shieldgemma-9b', 'shieldgemma-2b', 'gemma-4-26b-a4b-it', 'gemma-4-31b-it', 'gemma-3n-e4b-it', 'gemma-3n-e2b-it', 'gemma-3-4b-it', 'gemma-3-12b-it']

LLM backend: google
GenAI model (demos):     gemma-4-26b-a4b-it
Gatekeeper model (triage): gemma-4-26b-a4b-it
Embedding model (demo 6):   gemini-embedding-001
Shield model (safety pass): gemma-4-26b-a4b-it
Caps: pdfs/jur=3 pages/pdf=8 audio/jur=1 chunks/audio=4 thinking_budget=-1


In [24]:
# Re-print model catalog + resolved ids.
_log_llm_catalog = os.environ.get("GOVERNANCE_LOG_LLM_CATALOG", "0") == "1"

if USE_HF:
    from gemma_hf_backend import print_hf_model_catalog
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GENAI_MODEL, GATEKEEPER_MODEL), role="Hugging Face")
else:
    from gatekeeper_triage import print_available_models
    if _log_llm_catalog:
        print_available_models(
            _resolver_client,
            requested=(GENAI_MODEL, GATEKEEPER_MODEL, EMBEDDING_MODEL, SHIELD_MODEL),
            role="this API key",
        )
print("Resolved ids:")
print("  GENAI_MODEL     =", GENAI_MODEL)
print("  GATEKEEPER_MODEL=", GATEKEEPER_MODEL)
print("  EMBEDDING_MODEL =", EMBEDDING_MODEL)
print("  SHIELD_MODEL    =", SHIELD_MODEL)

Resolved ids:
  GENAI_MODEL     = gemma-4-26b-a4b-it
  GATEKEEPER_MODEL= gemma-4-26b-a4b-it
  EMBEDDING_MODEL = gemini-embedding-001
  SHIELD_MODEL    = gemma-4-26b-a4b-it


## 4) Step 0 — Gatekeeper Triage (data gate)

> **The Ledger of Influence — data gate.** Before any expensive deconstruction
> pass, route every raw input through `gatekeeper_triage.run_triage()`. The
> Gatekeeper sends each PDF and audio file to Gemma for a cheap, strict-JSON
> verdict (`is_governance_meeting`, `document_or_audio_type`, `confidence_score`,
> `reasoning`). Files the model rejects are moved to
> `01_raw_inputs/excluded_inputs/<STATE>/<scope>/<jurisdiction>/…` — the original
> geography subtree is replicated so we can audit *why* every file was kept or
> dropped.

Knobs:

| Env var | Default | Effect |
|---|---|---|
| `GOVERNANCE_GATEKEEPER_ENABLED` | `1` | Set `0` to skip the gate entirely. |
| `GOVERNANCE_GATEKEEPER_DRY_RUN` | `0` | Set `1` for a no-move audit pass — verdicts logged, nothing relocated. |
| `GOVERNANCE_GATEKEEPER_KINDS` | `pdf,audio` | Limit to one kind during demo runs. |
| `GOVERNANCE_GATEKEEPER_AUDIO_WINDOW` | `120` | Seconds of audio sent to triage (programmatic stream-depth cap). |
| `GOVERNANCE_GATEKEEPER_PDF_PAGES` | `2` | First N PDF pages sent to triage at HIGH (~1,120 image tokens). |
| `GOVERNANCE_GATEKEEPER_CONFIDENCE` | `0.6` | Minimum confidence to keep a file. |
| `GOVERNANCE_GATEKEEPER_MAX_FILES` | (unset) | Stop after N files — useful for live demos. |

Re-runs are idempotent: the walker prunes `excluded_inputs/` from its descent so
already-rejected files are never re-triaged.

To capture the verbose internal logging from `gatekeeper_triage.run_triage` and prevent it from cluttering the console, I will redirect its standard output and error streams to a log file in `03_processed_outputs/_gatekeeper/`. This ensures all detailed logs are recorded on Drive, making the console output cleaner for interactive use.

In [25]:
# Step 0 — Gatekeeper Triage. Routes raw inputs through Gemma before any deep pass.
import os
from pathlib import Path
import contextlib
import io # Needed for StringIO or file handling

import gatekeeper_triage

RAW_ROOT_FOR_GATE = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()

GATEKEEPER_ENABLED = os.environ.get("GOVERNANCE_GATEKEEPER_ENABLED", "1") != "0"
GATEKEEPER_DRY_RUN = os.environ.get("GOVERNANCE_GATEKEEPER_DRY_RUN", "0") == "1"

if not GATEKEEPER_ENABLED:
    print("Gatekeeper disabled (GOVERNANCE_GATEKEEPER_ENABLED=0). Skipping triage.")
elif not RAW_ROOT_FOR_GATE.is_dir():
    print(f"Skipping Gatekeeper: raw inputs root missing — {RAW_ROOT_FOR_GATE}")
else:
    kinds = tuple(
        k.strip().lower()
        for k in os.environ.get("GOVERNANCE_GATEKEEPER_KINDS", "pdf,audio").split(",")
        if k.strip()
    )
    max_files_env = os.environ.get("GOVERNANCE_GATEKEEPER_MAX_FILES", "").strip()
    max_files = int(max_files_env) if max_files_env else None

    # Persist the report alongside the other processed outputs for audit / demo.
    report_dir = PIPE.root / "03_processed_outputs" / "_gatekeeper"
    report_dir.mkdir(parents=True, exist_ok=True)
    from datetime import datetime, timezone
    import json as _json

    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path = report_dir / f"triage_console_log_{stamp}.txt"

    print(f"Gatekeeper triage console output will be logged to: {log_path.relative_to(PIPE.root)}")

    with open(log_path, 'w', encoding='utf-8') as f:
        with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
            print(f"Gatekeeper sweep | raw_root={RAW_ROOT_FOR_GATE} | kinds={kinds} | dry_run={GATEKEEPER_DRY_RUN}")
            gatekeeper_triage._configure_logging(verbose=False) # Keep internal module logging quiet
            report = gatekeeper_triage.run_triage(
                raw_root=RAW_ROOT_FOR_GATE,
                api_key=API_KEY,
                model=GATEKEEPER_MODEL,
                kinds=kinds,
                pdf_pages=int(os.environ.get("GOVERNANCE_GATEKEEPER_PDF_PAGES", "2")),
                audio_window_seconds=int(os.environ.get("GOVERNANCE_GATEKEEPER_AUDIO_WINDOW", "120")),
                confidence_threshold=float(os.environ.get("GOVERNANCE_GATEKEEPER_CONFIDENCE", "0.6")),
                dry_run=GATEKEEPER_DRY_RUN,
                max_files=max_files,
            )

    report_path = report_dir / f"triage_report_{stamp}.json"
    report_path.write_text(_json.dumps(report.to_dict(), indent=2, ensure_ascii=False), encoding="utf-8")

    print(
        f"\nGatekeeper done | proceed={len(report.proceed)} | excluded={len(report.excluded)} "
        f"| errors={len(report.errors)} | report → {report_path.relative_to(PIPE.root)}"
    )
    if report.excluded:
        print("\nExcluded files (mirrored under excluded_inputs/<state>/<scope>/<jurisdiction>/…):")
        for v in report.excluded[:10]:
            print(f"  · {v.relative_path}: {v.document_or_audio_type} ({v.confidence_score:.2f}) — {v.reasoning[:140]}")
        if len(report.excluded) > 10:
            print(f"  …and {len(report.excluded) - 10} more (see report).")

Gatekeeper triage console output will be logged to: 03_processed_outputs/_gatekeeper/triage_console_log_20260516_171826.txt


KeyboardInterrupt: 

### Modifying `gatekeeper_triage.py` for Meeting Date and Title Extraction

To implement the intelligent grouping logic, we need to instruct the Gatekeeper LLM to extract the `meeting_date` (in `YYYY-MM-DD` format) and a `meeting_title` during its triage process. This involves updating the system prompt and the expected JSON response schema within the `gatekeeper_triage.py` file. We also need to update the `TriageReportEntry` data structure to store these new pieces of information and modify its `from_model_response` method to parse them.

After executing the following cell, please **restart the Colab runtime** (Runtime -> Restart runtime) to ensure the updated `gatekeeper_triage` module is loaded.

In [ ]:
from pathlib import Path
import re

GATEKEEPER_TRIAGE_FILE_PATH = Path("/content/open-navigator/gatekeeper_triage.py")

# --- Define the new prompt and schema content ---

# New GATEKEEPER_SYSTEM_PROMPT with instructions for date and title extraction
NEW_GATEKEEPER_SYSTEM_PROMPT_SNIPPET = """You are a strict data gate for a local-government open-data pipeline. Before any expensive deconstruction pass, you route every raw input through Gemma for a cheap, strict-JSON verdict on whether it is a governance meeting record. You return `is_governance_meeting: true` only for files that are definitively local government meeting records (e.g., minutes, agendas, audio recordings of meetings, resolutions, ordinances). If you are uncertain or the file is not a meeting record, you return `false`. Always provide a `reasoning` field. For files that are identified as governance meeting records, you must also extract the precise meeting date in YYYY-MM-DD format and a concise, generic meeting title suitable for use in a folder name. The title should be unique enough to avoid conflicts on the same date but otherwise generic (e.g., 'City Council Meeting', 'Planning Commission', 'Public Hearing')."""

# New RESPONSE_SCHEMA_JSON including meeting_date and meeting_title
NEW_RESPONSE_SCHEMA_JSON_SNIPPET = '''{
    "type": "object",
    "properties": {
        "is_governance_meeting": {"type": "boolean", "description": "True if the file is a local government meeting record."},
        "document_or_audio_type": {"type": "string", "enum": ["minutes", "agenda", "ordinance", "resolution", "audio_transcript", "audio_recording", "other_governance_document", "not_governance"], "description": "The type of governance document or audio."},
        "confidence_score": {"type": "number", "minimum": 0.0, "maximum": 1.0, "description": "Confidence that the file is correctly classified."},
        "reasoning": {"type": "string", "description": "Brief explanation for the classification."},
        "meeting_date": {"type": ["string", "null"], "format": "date", "description": "The precise date of the meeting in YYYY-MM-DD format, if identifiable."},
        "meeting_title": {"type": ["string", "null"], "description": "A concise, generic title for the meeting (e.g., 'City Council Meeting', 'Planning Commission'), if identifiable."}
    },
    "required": ["is_governance_meeting", "document_or_audio_type", "confidence_score", "reasoning"]
}'''

# --- Read the original file content ---
original_content = GATEKEEPER_TRIAGE_FILE_PATH.read_text(encoding="utf-8")
modified_content = original_content

# --- Apply modifications ---

# 1. Update GATEKEEPER_SYSTEM_PROMPT
# Using regex to find and replace the entire string definition, allowing for multiline content.
# This regex looks for 'GATEKEEPER_SYSTEM_PROMPT = """' and then any characters (non-greedy) until '"""'
old_prompt_pattern = r"(GATEKEEPER_SYSTEM_PROMPT = """)(.*?)"""" # Capture the entire string literal
modified_content = re.sub(old_prompt_pattern, r'\1' + NEW_GATEKEEPER_SYSTEM_PROMPT_SNIPPET + r'"""', modified_content, flags=re.DOTALL)


# 2. Update RESPONSE_SCHEMA_JSON
# Similar approach, replacing the entire JSON string definition.
# This regex specifically looks for 'RESPONSE_SCHEMA_JSON = {' and then captures the entire JSON object.
old_schema_pattern = r"(RESPONSE_SCHEMA_JSON = )(\{.*?\}\n)" # Capturing the dict content itself
modified_content = re.sub(old_schema_pattern, r'\1' + NEW_RESPONSE_SCHEMA_JSON_SNIPPET + '\n', modified_content, flags=re.DOTALL)


# 3. Insert meeting_date and meeting_title into TriageReportEntry dataclass
# Find the line '    reasoning: str' and insert the new fields below it.
insert_dataclass_fields_pattern = r"(    reasoning: str\n)"
insert_dataclass_fields_replacement = r"\1    meeting_date: str | None = None # NEW\n    meeting_title: str | None = None # NEW\n"
modified_content = re.sub(insert_dataclass_fields_pattern, insert_dataclass_fields_replacement, modified_content)

# 4. Insert meeting_date and meeting_title into TriageReportEntry.from_model_response
# Find '            reasoning=model_output["reasoning"],' and insert the new assignments.
insert_from_model_response_pattern = r"(            reasoning=model_output\["reasoning"],\n)"
insert_from_model_response_replacement = r"\1            meeting_date=model_output.get("meeting_date"),\n            meeting_title=model_output.get("meeting_title"),\n"
modified_content = re.sub(insert_from_model_response_pattern, insert_from_model_response_replacement, modified_content)

# --- Write the modified content back to the file ---
GATEKEEPER_TRIAGE_FILE_PATH.write_text(modified_content, encoding="utf-8")

print(f"Successfully modified {GATEKEEPER_TRIAGE_FILE_PATH.name}. Please restart the Colab runtime for changes to take effect (Runtime -> Restart runtime).")

## 5) Walk `01_raw_inputs` by state / scope / jurisdiction

The walker honors the layout the sync script writes:
`01_raw_inputs/<STATE>/<county|municipality>/<jurisdiction_slug>/…` with FIPS
encoded in the slug (`county_01125` = Tuscaloosa County AL FIPS 01125;
`municipality_3006475` = Big Timber MT). Scraper-internal folders prefixed with
underscore (`_crawl_html`, `_sitemaps`, `_contact_images`) are skipped.

The cell below prints an inventory and stores it in `INVENTORIES` for the four
demo cells. Set `GOVERNANCE_RAW_INPUTS_ROOT` to point at a different root if your
raw inputs live outside the standard pipeline path.

In [ ]:
from pathlib import Path
import os

RAW_ROOT = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()
PROCESSED_ROOT = PIPE.root / "03_processed_outputs"
GEMMA_JSON_ROOT = PROCESSED_ROOT / "02_gemma_json"
SUMMARIES_ROOT = PROCESSED_ROOT / "03_human_summaries"
SCRATCH_AUDIO_ROOT = PROCESSED_ROOT / "_scratch_audio_chunks"

for p in (GEMMA_JSON_ROOT, SUMMARIES_ROOT, SCRATCH_AUDIO_ROOT):
    p.mkdir(parents=True, exist_ok=True)

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw inputs root missing: {RAW_ROOT}. Run 01_copy_scraped_meetings_cache_to_gdrive.py "
        "or set GOVERNANCE_RAW_INPUTS_ROOT."
    )

INVENTORIES = [inv for inv in walk_raw_inputs(RAW_ROOT) if inv.has_media]

print(f"Raw inputs root: {RAW_ROOT}")
print(f"Jurisdictions with media: {len(INVENTORIES)}")
for inv in INVENTORIES:
    j = inv.jurisdiction
    fips = j.fips or "—"
    print(
        f"  {j.relative_label}  fips={fips}  "
        f"pdfs={len(inv.pdfs)}  audio={len(inv.audio)}  images={len(inv.images)}"
    )

if not INVENTORIES:
    print(
        "\nNo media found. Drop PDFs / audio under "
        f"{RAW_ROOT}/<STATE>/<county|municipality>/<jurisdiction_slug>/…"
    )

## 6) Demo 1 — Native multimodality / visual document parsing

> **Dark-data problem:** Many municipal minutes from Tuscaloosa and Big Timber are
> physically printed, signed, and run through an office scanner — the resulting
> PDF contains **zero** digital text. Traditional text extractors return empty
> strings. Gemma 4 reads the page as an image and OCRs it natively.

For every PDF discovered by the walker, this cell:

1. Probes the digital-text layer with PyMuPDF (`pdftotext`-equivalent).
2. Tags PDFs with `<200` chars of extractable text as **scanned (dark-data)**.
3. Sends the PDF bytes to Gemma 4 with a minimal "extract everything" prompt.
4. Writes the model's transcribed text to
   `03_processed_outputs/02_gemma_json/<STATE>/<scope>/<jurisdiction>/<path>.visual_ocr.txt`.

The summary table front-loads the scanned-vs-digital split so judges immediately
see the dark-data win.

In [ ]:
# Demo 1 — Native multimodality / visual document parsing.
# Probes the digital-text layer of every PDF and uses Gemma 4 to OCR scans.
from datetime import datetime, timezone

DEMO1_SYSTEM = (
    "You are a careful document transcription engine. Faithfully transcribe every "
    "word and number on each page of the attached PDF in reading order. Preserve "
    "table structure with vertical bars and dashes. Do not paraphrase, summarize, "
    "or invent content."
)
DEMO1_USER = (
    "Transcribe the attached PDF page by page. Begin each page with a heading line "
    "'### Page <n>' on its own line. If a page is blank, write '(blank page)'."
)

demo1_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    if not pdfs:
        continue
    print(f"\n— {j.relative_label} — {len(pdfs)} PDF(s)")
    for pdf in pdfs:
        try:
            digital_chars = len(extract_pdf_digital_text(pdf))
        except Exception as e:
            print(f"  ! {pdf.name}: PDF probe failed — {e}")
            continue
        scanned = digital_chars < 200
        tag = "SCANNED (dark data)" if scanned else f"digital ({digital_chars} chars)"
        print(f"  • {pdf.name}: {tag}")

        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO1_SYSTEM,
                user_text=DEMO1_USER,
                media=[(pdf, "application/pdf")],
                temperature=0.0,
                max_output_tokens=8192,
                media_resolution=TOKEN_BUDGET_HIGH if scanned else None,
            )
        except Exception as e:
            print(f"    ! Gemma call failed: {e}")
            continue

        out_txt = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix=".visual_ocr.txt",
        )
        out_txt.write_text(result.text or "(empty response)", encoding="utf-8")
        demo1_report.append(
            {
                "jurisdiction": j.relative_label,
                "fips": j.fips,
                "pdf": str(pdf.relative_to(RAW_ROOT)),
                "scanned": scanned,
                "digital_chars": digital_chars,
                "output": str(out_txt.relative_to(PROCESSED_ROOT)),
                "model_chars": len(result.text or ""),
            }
        )
        print(f"    → {out_txt.relative_to(PROCESSED_ROOT)} ({len(result.text or '')} chars)")

scanned_count = sum(1 for r in demo1_report if r["scanned"])
print(
    f"\nDemo 1 done: {len(demo1_report)} PDFs, {scanned_count} flagged scanned "
    "→ Gemma 4 visual OCR recovered text that pdftotext could not."
)

import json as _json
demo1_summary_path = GEMMA_JSON_ROOT / "_demo1_visual_ocr_report.json"
demo1_summary_path.write_text(_json.dumps(demo1_report, indent=2), encoding="utf-8")
print("Report:", demo1_summary_path.relative_to(PROCESSED_ROOT))

## 7) Demo 2 — Adjustable visual token budget per page

> **Why:** Gemma 4 lets us spend up to ~1,120 image tokens per page when we need
> ultra-fine detail (financial ledgers, contract bid alignments, scanned forms),
> and drop to ~64 tokens per page on text-heavy minutes to protect runtime and
> API cost.

For each PDF the cell renders every page to a PNG, classifies it via
`classify_pdf_page_heuristic`, and routes:

- **`scanned`** → **HIGH** budget — needs visual OCR to recover any text at all.
- **`financial_or_tabular`** → **HIGH** budget — Big Timber courthouse architect
  rankings or airport paving contract awards demand column alignment fidelity.
- **`text_heavy`** → **LOW** budget — standard council minutes; speed + cost win.

Each page produces one JSON file under
`02_gemma_json/<mirrored>/<pdf_stem>/page_<n>.json` plus a per-PDF
`_token_budget_report.json` summarizing the budget split.

In [ ]:
# Demo 2 — Adjustable visual token budget per page.
# Classify each PDF page and route HIGH (~1,120 tokens) vs LOW (~64 tokens).
import json as _json
import time

DEMO2_SYSTEM = (
    "You are a careful page-level extractor. Return JSON only — no markdown fences."
)
DEMO2_USER_HIGH = (
    "This page contains tabular or financial content (bids, contract awards, "
    "ledgers, line-item budgets). Preserve column alignment and every dollar "
    "amount. Return JSON with this shape: "
    '{"page_type":"financial_or_tabular","raw_text":"...","line_items":[{"label":"...","amount":"..."}],"notes":"..."}'
)
DEMO2_USER_LOW = (
    "This page is standard meeting minutes text. Return JSON with this shape: "
    '{"page_type":"text_heavy","raw_text":"...","headline":"...","notes":"..."}'
)
DEMO2_USER_SCANNED = (
    "This page is a scanned image with no digital text. Visually OCR it. "
    "Return JSON with this shape: "
    '{"page_type":"scanned","raw_text":"...","notes":"..."}'
)
USER_BY_CLASS = {
    "financial_or_tabular": DEMO2_USER_HIGH,
    "text_heavy": DEMO2_USER_LOW,
    "scanned": DEMO2_USER_SCANNED,
}

demo2_report = []
_demo2_total_pdfs = sum(len(inv.pdfs[:MAX_PDFS_PER_JUR]) for inv in INVENTORIES)
_demo2_jurs_with_pdfs = sum(1 for inv in INVENTORIES if inv.pdfs[:MAX_PDFS_PER_JUR])
print(
    f"Demo 2 scope: {len(INVENTORIES)} inventories, "
    f"{_demo2_jurs_with_pdfs} with PDFs, "
    f"{_demo2_total_pdfs} PDFs after cap MAX_PDFS_PER_JUR={MAX_PDFS_PER_JUR}."
)
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    for pdf in pdfs:
        print(f"\n— {j.relative_label} / {pdf.name}")
        try:
            pages = render_pdf_pages(pdf, dpi=200)
        except Exception as e:
            print(f"  ! render failed: {e}")
            continue
        pages = pages[:MAX_PAGES_PER_PDF]

        per_pdf_dir = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix="",
        )
        per_pdf_dir.mkdir(parents=True, exist_ok=True)
        if per_pdf_dir.is_file():
            per_pdf_dir.unlink()
            per_pdf_dir.mkdir(parents=True, exist_ok=True)

        pdf_summary = {
            "jurisdiction": j.relative_label,
            "fips": j.fips,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "page_count": len(pages),
            "budget_split": {TOKEN_BUDGET_HIGH: 0, TOKEN_BUDGET_LOW: 0},
            "pages": [],
        }
        for page in pages:
            budget = page.token_budget
            user = USER_BY_CLASS.get(page.classification, DEMO2_USER_LOW)
            t0 = time.time()
            try:
                result = call_google_genai_multimodal(
                    api_key=API_KEY,
                    model=GENAI_MODEL,
                    system_instruction=DEMO2_SYSTEM,
                    user_text=user,
                    media=[(page.image_bytes, "image/png")],
                    temperature=0.0,
                    max_output_tokens=2048,
                    media_resolution=budget,
                )
            except Exception as e:
                print(f"  ! page {page.page_index + 1}: Gemma call failed — {e}")
                continue
            elapsed = time.time() - t0

            try:
                page_json = _json.loads(result.text.strip().lstrip("`"))
            except Exception:
                page_json = {"_parse_error": True, "_raw": result.text[:2000]}

            page_out = per_pdf_dir / f"page_{page.page_index + 1:03d}.json"
            page_out.write_text(
                _json.dumps(
                    {
                        "page_index": page.page_index,
                        "classification": page.classification,
                        "token_budget": budget,
                        "elapsed_s": round(elapsed, 2),
                        "model": GENAI_MODEL,
                        "extracted": page_json,
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            pdf_summary["budget_split"][budget] = pdf_summary["budget_split"].get(budget, 0) + 1
            pdf_summary["pages"].append(
                {
                    "page": page.page_index + 1,
                    "classification": page.classification,
                    "token_budget": budget,
                    "elapsed_s": round(elapsed, 2),
                }
            )
            print(
                f"  page {page.page_index + 1}: {page.classification:>22}  "
                f"→ {budget:<6} ({elapsed:.1f}s)"
            )

        report_path = per_pdf_dir / "_token_budget_report.json"
        report_path.write_text(_json.dumps(pdf_summary, indent=2), encoding="utf-8")
        demo2_report.append(pdf_summary)
        high = pdf_summary["budget_split"].get(TOKEN_BUDGET_HIGH, 0)
        low = pdf_summary["budget_split"].get(TOKEN_BUDGET_LOW, 0)
        print(f"  budget split: HIGH={high}  LOW={low}  → report {report_path.relative_to(PROCESSED_ROOT)}")

if demo2_report:
    total_high = sum(r["budget_split"].get(TOKEN_BUDGET_HIGH, 0) for r in demo2_report)
    total_low = sum(r["budget_split"].get(TOKEN_BUDGET_LOW, 0) for r in demo2_report)
    total = total_high + total_low or 1
    print(
        f"\nDemo 2 done: {total} pages across {len(demo2_report)} PDF(s). "
        f"HIGH-budget pages: {total_high} ({100 * total_high // total}%). "
        f"LOW-budget pages: {total_low} ({100 * total_low // total}%). "
        "Dropping text-heavy minutes to LOW is the cost / latency win we show judges."
    )
else:
    print("\nDemo 2: no PDFs processed (check MAX_PDFS_PER_JUR and inventory above).")

## 8) Demo 3 — Built-in thinking on the deconstruction prompt

> **Hackathon talking point:** April 2026 Tuscaloosa County nuisance property
> demolitions. Gemma 4's multi-step planning surfaces the prevailing narrative
> (collective safety + blight removal) and the dissenting diagnosis (individual
> property rights from protesting neighbors) in the same pass — not just a
> chronological summary.

For each jurisdiction the cell picks a representative PDF (largest by page count,
preferring filenames that suggest minutes / hearings) and runs the full
`prompts/policy_analysis_v1.md` prompt with `include_thoughts=True` and an
extended `thinking_budget`. Outputs:

- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.json` — parsed JSON analysis.
- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.raw.txt` — raw model output.
- `…/02_gemma_json/<mirrored>/<pdf_stem>.thinking.thoughts.md` — reasoning trace.
- `…/03_human_summaries/<mirrored>/<pdf_stem>.thinking.summary.md` — human summary.

In [ ]:
# Demo 3 — Built-in thinking mode on the deconstruction prompt.
# One representative PDF per jurisdiction is processed with include_thoughts=True.
import json as _json

DEMO3_SYSTEM = (
    "You are an expert political scientist and data architect specializing in "
    "local governance. Follow the user's instructions exactly; preserve JSON validity."
)

_PRIORITY_PATTERNS = (
    "demolition", "demolitions", "nuisance",
    "minutes", "regular_session", "regular-session", "regular session",
    "council", "commission", "board", "hearing",
)


def _pick_representative_pdf(pdfs):
    if not pdfs:
        return None
    scored = []
    for p in pdfs:
        name = p.name.lower()
        score = 0
        for tag in _PRIORITY_PATTERNS:
            if tag in name:
                score += 5
        try:
            score += min(p.stat().st_size // 50_000, 50)
        except OSError:
            pass
        scored.append((score, p))
    scored.sort(key=lambda t: (-t[0], t[1].name))
    return scored[0][1]


demo3_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdf = _pick_representative_pdf(inv.pdfs)
    if pdf is None:
        continue
    print(f"\n— {j.relative_label}: {pdf.name}")

    geo_hint = (
        f"Geography hint from folder layout: state_code={j.state_code}, "
        f"scope={j.scope}, fips_or_place_id={j.fips or 'unknown'}. "
        "Use this when populating county_fips / county / postal_code in each decision."
    )
    user_text = (
        f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
        "The attached PDF contains the meeting record. Apply the full deconstruction "
        "prompt to it. Stick to what is actually in the document."
    )

    try:
        result = call_google_genai_multimodal(
            api_key=API_KEY,
            model=GENAI_MODEL,
            system_instruction=DEMO3_SYSTEM,
            user_text=user_text,
            media=[(pdf, "application/pdf")],
            temperature=0.1,
            max_output_tokens=8192,
            media_resolution=TOKEN_BUDGET_HIGH,
            include_thoughts=True,
            thinking_budget=THINKING_BUDGET,
        )
    except Exception as e:
        print(f"  ! Gemma call failed: {e}")
        continue

    parsed = parse_policy_analysis_response(result.text or "")

    raw_out = mirror_output_path(
        input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
        suffix=".thinking.raw.txt",
    )
    raw_out.write_text(result.text or "", encoding="utf-8")

    if parsed.get("json_analysis") is not None:
        json_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.json",
        )
        json_out.write_text(
            _json.dumps(parsed["json_analysis"], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"  → {json_out.relative_to(PROCESSED_ROOT)}")

    if parsed.get("summary"):
        summary_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=SUMMARIES_ROOT,
            suffix=".thinking.summary.md",
        )
        summary_out.write_text(parsed["summary"], encoding="utf-8")
        print(f"  → {summary_out.relative_to(PROCESSED_ROOT)}")

    if result.thoughts:
        thoughts_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.thoughts.md",
        )
        thoughts_out.write_text(result.thoughts, encoding="utf-8")
        print(
            f"  → {thoughts_out.relative_to(PROCESSED_ROOT)} "
            f"(reasoning trace: {len(result.thoughts)} chars)"
        )
    else:
        print("  (no thoughts returned — model may not have surfaced a trace this run)")

    demo3_report.append(
        {
            "jurisdiction": j.relative_label,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "thoughts_chars": len(result.thoughts or ""),
            "json_ok": parsed.get("json_analysis") is not None and "_error" not in (parsed.get("json_analysis") or {}),
            "parse_error": parsed.get("parse_error"),
        }
    )

if demo3_report:
    ok = sum(1 for r in demo3_report if r["json_ok"])
    print(
        f"\nDemo 3 done: {len(demo3_report)} PDFs deconstructed, "
        f"{ok} produced parseable JSON. Reasoning trace saved alongside each output."
    )
else:
    print("\nDemo 3: no PDFs available — check the walker output above.")

## 9) Demo 4 — Long-meeting chunking + Policy Drift Detector

> **Why:** Governance meeting videos often run 2–4 hours. We feed the audio in
> **15-minute chunks** to Gemma 4 and rely on the model's alternating local
> sliding-window + global attention to keep the broader thread across the meeting.
> The drift detector then runs a second Gemma pass that takes every chunk's JSON
> output and reports how a specific subject (e.g. Tuscaloosa downtown zoning)
> migrates from origination to final amendment.

For each audio file the cell:

1. Splits the file with `ffmpeg` into 15-minute MP3 chunks under a scratch dir.
2. Runs `prompts/policy_analysis_v1.md` per chunk, attaching the audio chunk.
3. Stores per-chunk JSON under `02_gemma_json/<mirrored>/<audio_stem>/chunk_<n>.json`.
4. Calls `policy_drift_summarize` with the assembled chunk JSONs and writes
   `…/<audio_stem>/policy_drift.json` (+ `.mmd` Mermaid timeline).

Set `GOVERNANCE_DRIFT_FOCUS` to bias the detector toward a specific subject string.

In [ ]:
# Demo 4 — Long-meeting chunking + Policy Drift Detector.
# Splits audio into 15-minute chunks, runs policy_analysis_v1 per chunk, then runs the drift pass.
import json as _json

DEMO4_SYSTEM = (
    "You are an expert political scientist analyzing one chunk of a long meeting. "
    "Follow the user's instructions exactly; preserve JSON validity. The chunk_index "
    "tells you which 15-minute slice of the meeting this audio covers."
)

demo4_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    audios = inv.audio[:MAX_AUDIO_PER_JUR]
    if not audios:
        continue
    print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
    for audio in audios:
        print(f"  • {audio.name}")
        per_audio_dir = mirror_output_path(
            input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
        )
        per_audio_dir.mkdir(parents=True, exist_ok=True)

        # Scratch dir for ffmpeg chunks. Mirror layout so multiple meetings don't collide.
        rel = audio.resolve().relative_to(RAW_ROOT.resolve())
        scratch = SCRATCH_AUDIO_ROOT / rel.with_suffix("")
        scratch.mkdir(parents=True, exist_ok=True)

        try:
            chunks = chunk_audio_ffmpeg(audio, out_dir=scratch, chunk_minutes=15, fmt="mp3")
        except Exception as e:
            print(f"    ! ffmpeg chunking failed: {e}")
            continue
        chunks = chunks[:MAX_AUDIO_CHUNKS]
        print(f"    {len(chunks)} chunk(s) of 15 min each (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")

        chunk_jsons = []
        for idx, chunk_path in enumerate(chunks):
            geo_hint = (
                f"Geography hint: state_code={j.state_code}, scope={j.scope}, "
                f"fips_or_place_id={j.fips or 'unknown'}. chunk_index={idx} of {len(chunks)}."
            )
            user_text = (
                f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
                "The attached audio is one 15-minute slice of a longer governance meeting. "
                "Apply the deconstruction prompt to what is audible. Use the chunk_index "
                "to anchor the timeline and assign consistent subject_id slugs across chunks."
            )
            try:
                result = call_google_genai_multimodal(
                    api_key=API_KEY,
                    model=GENAI_MODEL,
                    system_instruction=DEMO4_SYSTEM,
                    user_text=user_text,
                    media=[(chunk_path, mime_for(chunk_path))],
                    temperature=0.1,
                    max_output_tokens=8192,
                )
            except Exception as e:
                print(f"    ! chunk {idx} failed: {e}")
                continue
            parsed = parse_policy_analysis_response(result.text or "")
            chunk_out = per_audio_dir / f"chunk_{idx:03d}.json"
            chunk_out.write_text(
                _json.dumps(
                    {
                        "chunk_index": idx,
                        "audio_source": str(chunk_path.name),
                        "json_analysis": parsed.get("json_analysis"),
                        "summary": parsed.get("summary"),
                        "parse_error": parsed.get("parse_error"),
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            chunk_jsons.append(parsed.get("json_analysis") or {})
            print(f"    chunk {idx}: → {chunk_out.relative_to(PROCESSED_ROOT)}")

        if not chunk_jsons:
            continue

        # Drift detector: Gemma 4 takes every chunk's JSON and reports per-subject
        # narrative drift along the five `narrative_analysis` dimensions defined in
        # `prompts/policy_analysis_v1.md` (dominant_narrative, dissenting_interpretations,
        # causal_interpretations, value_conflicts, tradeoff_analysis). The canonical
        # prompt body is pinned into context so the model honors the latest schema verbatim.
        drift = policy_drift_summarize(
            chunk_jsons,
            api_key=API_KEY,
            model=GENAI_MODEL,
            focus_hint=DRIFT_FOCUS,
            canonical_prompt_text=POLICY_PROMPT,
        )
        drift_out = per_audio_dir / "policy_drift.json"
        drift_out.write_text(_json.dumps(drift, indent=2, ensure_ascii=False), encoding="utf-8")

        drifted = drift.get("subjects") or drift.get("drifted_subjects") or []

        # Concatenate per-subject Mermaid timelines so the .mmd file is a single
        # diagram per audio (legacy schema returned one top-level diagram_timeline).
        mmd_blocks = []
        for s in drifted:
            tl = s.get("diagram_timeline")
            if isinstance(tl, str) and tl.strip():
                label = s.get("subject_label") or s.get("subject_id") or "subject"
                mmd_blocks.append(f"%% {label}\n{tl}")
        legacy_tl = drift.get("diagram_timeline")
        if not mmd_blocks and isinstance(legacy_tl, str) and legacy_tl.strip():
            mmd_blocks.append(legacy_tl)
        if mmd_blocks:
            (per_audio_dir / "policy_drift.mmd").write_text(
                "\n\n".join(mmd_blocks), encoding="utf-8"
            )

        meeting_summary = drift.get("meeting_level_summary") or {}
        print(
            f"    drift detector: {len(drifted)} subject(s) tracked across {len(chunk_jsons)} chunks "
            f"→ {drift_out.relative_to(PROCESSED_ROOT)}"
        )
        if meeting_summary.get("headline"):
            print(f"      meeting headline: {meeting_summary['headline'][:140]}")
        for s in drifted[:5]:
            label = s.get("subject_label") or s.get("subject_id") or "?"
            headline = s.get("drift_headline") or s.get("drift_summary") or ""
            stability = (s.get("narrative_stability_assessment") or {}).get("narrative_novelty")
            stability_tag = f" [{stability}]" if stability else ""
            print(f"      · {label}{stability_tag}: {headline[:120]}")
        demo4_report.append(
            {
                "jurisdiction": j.relative_label,
                "audio": str(audio.relative_to(RAW_ROOT)),
                "chunks": len(chunk_jsons),
                "subjects_tracked": len(drifted),
                "subjects_with_drift": sum(1 for s in drifted if s.get("drift_observed")),
                "emergent_value_tensions": meeting_summary.get("emergent_value_tensions") or [],
            }
        )

if demo4_report:
    print(
        f"\nDemo 4 done: {len(demo4_report)} audio file(s) processed. "
        "Gemma's alternating local + global attention kept subject ids consistent "
        "across 15-minute chunks; drift detector reported per-subject drift along the "
        "five narrative_analysis dimensions from policy_analysis_v1.md."
    )
else:
    print("\nDemo 4: no audio files in the inventory. Drop .mp3/.mp4/.wav under a jurisdiction folder.")

## 9a) Demo 4a — Plain Transcription (English / Spanish)

> **Why:** Demo 4 emits *analyzed* JSON per audio chunk (policy_analysis_v1
> deconstruction). For provenance — and for judges who want to read the literal
> words spoken — we run a second, cheaper pass that returns *only* the
> transcribed text. The transcript is a citable artifact for the Ledger of
> Influence: "this claim came from the audio at minute 47."

For each audio file the cell:

1. Reuses (or generates) the same 15-minute chunks Demo 4 writes under
   `_scratch_audio_chunks/`.
2. Runs `transcribe_audio_with_gemma(language=...)` per chunk.
3. Writes one transcript per language to
   `02_gemma_json/<mirrored>/<audio_stem>/transcript.<lang>.txt`.

| Env var | Default | Effect |
|---|---|---|
| `GOVERNANCE_TRANSCRIPTION_LANGUAGES` | `en` | Comma-separated language tags. Supported: `en`, `es`. Set `en,es` to run both. |
| `GOVERNANCE_TRANSCRIPTION_MAX_CHUNKS` | _(unset → uses `MAX_AUDIO_CHUNKS`)_ | Per-file chunk cap for transcription only. |

The prompt is the literal hackathon template — *Transcribe the following speech
segment in {LANGUAGE} into {LANGUAGE} text* — with `temperature=0.0` and no
thinking config, so output is deterministic. Newlines emitted by the model are
collapsed defensively so each chunk's transcript is a single line, joined with
spaces across chunks.

In [ ]:
# Demo 4a — Plain Transcription per audio file.
# Mirrors Demo 4's audio iteration so transcripts land beside the chunk JSONs.
_TRANSCRIPTION_LANGS_RAW = os.environ.get("GOVERNANCE_TRANSCRIPTION_LANGUAGES", "en")
TRANSCRIPTION_LANGS = [
    lang.strip()
    for lang in _TRANSCRIPTION_LANGS_RAW.split(",")
    if lang.strip()
]
# Validate up-front so a typo in the env var fails fast, not after N audio calls.
_SUPPORTED_LANG_TAGS = {tag.lower() for tag in TRANSCRIPTION_SUPPORTED_LANGUAGES.values()}
_invalid_langs = [
    lang for lang in TRANSCRIPTION_LANGS if lang.lower() not in _SUPPORTED_LANG_TAGS
]
if _invalid_langs:
    raise RuntimeError(
        f"GOVERNANCE_TRANSCRIPTION_LANGUAGES contains unsupported tag(s) {_invalid_langs}. "
        f"Supported: {sorted(_SUPPORTED_LANG_TAGS)} (currently {list(TRANSCRIPTION_SUPPORTED_LANGUAGES)})."
    )

_TRANSCRIPTION_MAX_CHUNKS = int(
    os.environ.get("GOVERNANCE_TRANSCRIPTION_MAX_CHUNKS", str(MAX_AUDIO_CHUNKS))
)

demo4a_report = []
if not TRANSCRIPTION_LANGS:
    print("Demo 4a: GOVERNANCE_TRANSCRIPTION_LANGUAGES is empty — skipping.")
else:
    print(f"Demo 4a: transcribing in {TRANSCRIPTION_LANGS} (cap {_TRANSCRIPTION_MAX_CHUNKS} chunks/file)")
    for inv in INVENTORIES:
        j = inv.jurisdiction
        audios = inv.audio[:MAX_AUDIO_PER_JUR]
        if not audios:
            continue
        print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
        for audio in audios:
            print(f"  • {audio.name}")
            per_audio_dir = mirror_output_path(
                input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
            )
            per_audio_dir.mkdir(parents=True, exist_ok=True)

            rel = audio.resolve().relative_to(RAW_ROOT.resolve())
            scratch = SCRATCH_AUDIO_ROOT / rel.with_suffix("")
            scratch.mkdir(parents=True, exist_ok=True)

            # Reuse Demo 4 chunks if they're already on disk; otherwise chunk now.
            existing = sorted(scratch.glob("chunk_*.mp3"))
            if existing:
                chunks = existing
                print(f"    reusing {len(chunks)} existing chunk(s) from {scratch.name}/")
            else:
                try:
                    chunks = chunk_audio_ffmpeg(audio, out_dir=scratch, chunk_minutes=15, fmt="mp3")
                except Exception as e:
                    print(f"    ! ffmpeg chunking failed: {e}")
                    continue
            chunks = chunks[:_TRANSCRIPTION_MAX_CHUNKS]
            print(f"    {len(chunks)} chunk(s) to transcribe")

            for lang in TRANSCRIPTION_LANGS:
                pieces = []
                errors = 0
                for idx, chunk_path in enumerate(chunks):
                    try:
                        text = transcribe_audio_with_gemma(
                            api_key=API_KEY,
                            model=GENAI_MODEL,
                            audio_path=chunk_path,
                            language=lang,
                            mime_type=mime_for(chunk_path),
                        )
                    except Exception as e:
                        errors += 1
                        print(f"    ! chunk {idx} [{lang}] failed: {e}")
                        continue
                    if text:
                        pieces.append(text)

                if not pieces:
                    print(f"    [{lang}] no text produced — skipping write")
                    continue

                # Join chunk transcripts with single spaces. The model is instructed
                # not to emit newlines; we still collapse defensively in the helper.
                transcript = " ".join(pieces)
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                out_path.write_text(transcript, encoding="utf-8")
                print(
                    f"    [{lang}] → {out_path.relative_to(PROCESSED_ROOT)} "
                    f"({len(transcript):,} chars from {len(pieces)}/{len(chunks)} chunks"
                    + (f", {errors} error(s)" if errors else "")
                    + ")"
                )
                demo4a_report.append(
                    {
                        "jurisdiction": j.relative_label,
                        "audio": str(audio.relative_to(RAW_ROOT)),
                        "language": lang,
                        "chunks_transcribed": len(pieces),
                        "chunks_attempted": len(chunks),
                        "errors": errors,
                        "chars": len(transcript),
                    }
                )

if demo4a_report:
    print(
        f"\nDemo 4a done: {len(demo4a_report)} transcript file(s) written. "
        "Plain text is now beside each audio's chunk JSONs for citation / human review."
    )
else:
    print("\nDemo 4a: no transcripts written (no audio in inventory or all chunks failed).")

## 10) Demo 5 — Contact image enrichment (people vs. subjects)

> Each jurisdiction folder may carry a ``_contact_images/`` directory of scraped
> contact photos (council members, department heads, public officials). The walker
> exposes those images via ``inv.images``. This cell sends each image to Gemma 4
> vision with a strict-JSON prompt:
>
> - **If the image is a person** (public official in a public-records context):
>   return perceived ``age_range``, ``race``, ``gender``, ``ethnicity``, and
>   ``demeanor`` (e.g. ``formal``, ``smiling``, ``neutral``). All fields are
>   explicitly *perceived* — the model is instructed to return ``null`` rather
>   than guess when the image is too low-resolution, partially occluded, or
>   ambiguous, and to include a confidence score.
> - **If the image is not a person** (logo, building, document scan, map, chart,
>   blank): return a brief ``subject_tag`` describing what it is.
>
> Outputs mirror the input layout under
> ``03_processed_outputs/02_gemma_json/<STATE>/<scope>/<jurisdiction>/_contact_images/<name>.json``.

In [ ]:
# Demo 5 — Contact image enrichment. For each image found under a jurisdiction,
# decide whether it is a person; return perceived demographics or a subject tag.
import json as _json

DEMO5_SYSTEM = (
    "You are a careful image triage assistant for a local-government open-data "
    "pipeline. You analyze public contact photos of elected officials and "
    "department heads scraped from public government websites. You always return "
    "strict JSON only. You never claim certainty about demographic attributes — "
    "every attribute is explicitly perceived from the image and you return null "
    "when the image is too low-resolution, partially occluded, ambiguous, or when "
    "you would be guessing rather than observing."
)
DEMO5_USER = (
    "Look at the attached image and decide whether it depicts a single identifiable "
    "person. Return JSON with this shape:\n\n"
    "{\n"
    "  \"is_person\": bool,\n"
    "  \"confidence\": float between 0.0 and 1.0,\n"
    "  \"perceived\": {\n"
    "    \"age_range\": \"one of: child, teen, 20-29, 30-39, 40-49, 50-59, 60-69, 70-plus, or null\",\n"
    "    \"race\": \"perceived racial category or null\",\n"
    "    \"gender\": \"perceived gender presentation or null\",\n"
    "    \"ethnicity\": \"perceived ethnic background or null\",\n"
    "    \"demeanor\": \"one of: formal, smiling, neutral, stern, candid, or null\"\n"
    "  } or null when is_person is false,\n"
    "  \"subject_tag\": \"short descriptor when is_person is false (e.g. 'city logo', "
    "'aerial photo of courthouse', 'organizational chart'); otherwise null\",\n"
    "  \"notes\": \"short caveat — e.g. 'low resolution', 'side profile', "
    "'multiple people visible', 'official portrait crop'\"\n"
    "}\n\n"
    "Rules:\n"
    "- These are PUBLIC officials in a PUBLIC-records transparency context.\n"
    "- Every 'perceived.*' field is a best visual estimate, not a factual claim.\n"
    "- Return null for any 'perceived' subfield where you would be guessing.\n"
    "- If multiple people appear, set is_person=false and use subject_tag='group photo' (and notes).\n"
    "- Return only the JSON object, no prose or markdown."
)


def _mime_for_image(p):
    ext = p.suffix.lower()
    if ext in (".jpg", ".jpeg"): return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext == ".webp": return "image/webp"
    if ext == ".gif": return "image/gif"
    if ext == ".bmp": return "image/bmp"
    return "application/octet-stream"


MAX_IMAGES_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_IMAGES_PER_JUR", "12"))

demo5_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    if not inv.images:
        continue
    images = inv.images[:MAX_IMAGES_PER_JUR]
    print(f"\n— {j.relative_label}: {len(images)} image(s) (cap={MAX_IMAGES_PER_JUR})")
    for img in images:
        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO5_SYSTEM,
                user_text=DEMO5_USER,
                media=[(img, _mime_for_image(img))],
                temperature=0.0,
                max_output_tokens=1024,
                media_resolution=TOKEN_BUDGET_HIGH,
            )
        except Exception as e:
            print(f"  ! {img.name}: Gemma call failed — {e}")
            continue

        try:
            payload = _json.loads((result.text or "").strip().lstrip("`"))
        except Exception:
            payload = {"_parse_error": True, "_raw": (result.text or "")[:2000]}

        out_path = mirror_output_path(
            input_path=img, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".image_triage.json",
        )
        out_path.write_text(
            _json.dumps(
                {
                    "image": str(img.relative_to(RAW_ROOT)),
                    "jurisdiction_id": j.jurisdiction_id,
                    "model": GENAI_MODEL,
                    "result": payload,
                },
                indent=2, ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        is_person = bool(payload.get("is_person")) if isinstance(payload, dict) else False
        if is_person:
            perceived = payload.get("perceived") or {}
            details = ", ".join(
                f"{k}={v}" for k, v in perceived.items() if v not in (None, "")
            ) or "(no observable attributes)"
            print(f"  · {img.name}: PERSON — {details}")
        else:
            tag = payload.get("subject_tag") if isinstance(payload, dict) else None
            print(f"  · {img.name}: non-person → {tag or '(no tag)'}")
        demo5_report.append(
            {
                "jurisdiction": j.relative_label,
                "jurisdiction_id": j.jurisdiction_id,
                "image": str(img.relative_to(RAW_ROOT)),
                "is_person": is_person,
                "output": str(out_path.relative_to(PROCESSED_ROOT)),
            }
        )

if demo5_report:
    persons = sum(1 for r in demo5_report if r["is_person"])
    print(
        f"\nDemo 5 done: {len(demo5_report)} images classified across "
        f"{len({r['jurisdiction'] for r in demo5_report})} jurisdictions. "
        f"{persons} persons / {len(demo5_report) - persons} non-person subjects."
    )
else:
    print("\nDemo 5: no images in the inventory (check _contact_images/ folders).")

## 11) Optional — merge per-jurisdiction reference data by `jurisdiction_id`

Two parallel reference buckets live under `02_reference_data/`:

- **`meeting_data_by_jurisdiction_id/`** — Orbis exports, registry rows, and any
  other per-jurisdiction reference material (JSON lookups + PDFs sit side-by-side).
  Attached to each analysis under `meeting_data_profile`.
- **`contacts_by_jurisdiction_id/`** — scraped / curated contact rosters keyed
  by `jurisdiction_id` (never `org_id`). Attached under `contacts_profile`.

The cell walks every `*.json` produced by Demo 2 and Demo 3, derives the
`jurisdiction_id` from the mirrored output path, and runs both merges. Any file
matching `*_lookup_by_jurisdiction_id.json` in either bucket is picked up; PDFs
and other artifacts in those folders are left alone as human-readable reference.

In [ ]:
# Walk every *.json under GEMMA_JSON_ROOT and attach the matching meeting-data
# and contacts rows by jurisdiction_id, derived from the mirrored output path.
import json as _json


def _jurisdiction_id_from_output(path):
    """Mirror layout is <STATE>/<scope>/<jur_slug>/...  →  jurisdiction_<state>_<scope>_<tail>."""
    try:
        rel = path.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        return None
    parts = rel.parts
    if len(parts) < 3:
        return None
    state, scope, slug = parts[0].lower(), parts[1].lower(), parts[2].lower()
    scope_prefix = f"{scope}_"
    tail = slug[len(scope_prefix):] if slug.startswith(scope_prefix) else slug
    return f"jurisdiction_{state}_{scope}_{tail}"


meeting_data = load_meeting_data_lookup(PIPE.meeting_data_by_jurisdiction_id)
contacts = load_contacts_lookup(PIPE.contacts_by_jurisdiction_id)

if not meeting_data and not contacts:
    print(
        f"No reference lookups found under {PIPE.meeting_data_by_jurisdiction_id} "
        f"or {PIPE.contacts_by_jurisdiction_id} — skip."
    )
else:
    print(
        f"Loaded {len(meeting_data)} meeting_data and {len(contacts)} contacts "
        "rows keyed by jurisdiction_id."
    )
    merged_meeting = 0
    merged_contacts = 0
    missing_both = 0
    skipped = 0
    for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
        if jp.name.startswith("_") or jp.name.startswith("policy_drift"):
            continue
        try:
            data = _json.loads(jp.read_text(encoding="utf-8"))
        except _json.JSONDecodeError:
            skipped += 1
            continue
        jid = _jurisdiction_id_from_output(jp)
        if not jid:
            continue
        analysis = data
        if isinstance(data, dict) and "json_analysis" in data and isinstance(data["json_analysis"], dict):
            analysis = data["json_analysis"]
        if not isinstance(analysis, dict):
            continue
        had_meeting = jid in meeting_data
        had_contacts = jid in contacts
        if not (had_meeting or had_contacts):
            missing_both += 1
            continue
        if had_meeting:
            merge_meeting_data_by_jurisdiction(analysis, meeting_data, jid)
            merged_meeting += 1
        if had_contacts:
            merge_contacts_by_jurisdiction(analysis, contacts, jid)
            merged_contacts += 1
        jp.write_text(_json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tags = []
        if had_meeting:
            tags.append("meeting_data")
        if had_contacts:
            tags.append("contacts")
        print(f"  enriched: {jp.relative_to(PROCESSED_ROOT)}  jurisdiction_id={jid}  +{','.join(tags)}")
    print(
        f"\nMerge done: meeting_data={merged_meeting}, contacts={merged_contacts}, "
        f"no-match jurisdictions={missing_both}, unparseable files={skipped}."
    )

## 12) Demo 6 — EmbeddingGemma cross-jurisdiction clustering

Walks every JSON in `03_processed_outputs/02_gemma_json/`, extracts the high-signal text snippets (decision titles, agenda subjects, raw_text headlines, drift summaries), embeds each with **EmbeddingGemma**, and reports the top similar pairs **across jurisdictions**. The goal: surface that the same policy theme (e.g. *downtown zoning amendments*) is being debated in both Tuscaloosa and Big Timber, even though each meeting record uses different vocabulary.

Outputs:

- `…/04_embeddings/items.jsonl` — one row per embedded snippet (jurisdiction, source path, text, vector dimensions).
- `…/04_embeddings/cross_jurisdiction_pairs.json` — top-N pairs above the similarity threshold, scoped to **different** jurisdictions.

Tunables (env):

| Env var | Default | What it does |
|---|---|---|
| `GOVERNANCE_EMBEDDING_MODEL` | `embeddinggemma-300m` | EmbeddingGemma alias from AI Studio |
| `GOVERNANCE_EMBED_MAX_ITEMS` | `200` | Hard cap so a free-tier judging run can't blow through quota |
| `GOVERNANCE_EMBED_SIM_THRESHOLD` | `0.7` | Cosine threshold for the cross-jurisdiction pairs report |

In [ ]:
# Demo 6 — EmbeddingGemma: cluster similar policy items across jurisdictions.
import json as _json

EMBED_MAX_ITEMS = int(os.environ.get("GOVERNANCE_EMBED_MAX_ITEMS", "200"))
EMBED_SIM_THRESHOLD = float(os.environ.get("GOVERNANCE_EMBED_SIM_THRESHOLD", "0.7"))
EMBED_REPORT_TOP_N = int(os.environ.get("GOVERNANCE_EMBED_REPORT_TOP_N", "30"))


def _harvest_snippets(json_path):
    """Pull high-signal text snippets from one Gemma JSON output."""
    try:
        data = _json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return []
    snippets = []
    payload = data
    if isinstance(data, dict) and isinstance(data.get("json_analysis"), dict):
        payload = data["json_analysis"]
    if not isinstance(payload, dict):
        return []
    # Demo 1 + 2 outputs carry raw_text / headlines.
    if isinstance(payload.get("raw_text"), str) and len(payload["raw_text"]) > 40:
        snippets.append(payload["raw_text"][:600])
    if isinstance(payload.get("headline"), str):
        snippets.append(payload["headline"])
    # Demo 3 outputs: subjects[], decisions[].
    for sub in payload.get("subjects", []) or []:
        name = sub.get("name") or sub.get("descriptive_name") or sub.get("subject_id")
        if name:
            snippets.append(str(name))
    for dec in payload.get("decisions", []) or []:
        title = dec.get("decision_title") or dec.get("title")
        if title:
            snippets.append(str(title))
    return [s.strip() for s in snippets if s and s.strip()]


items = []
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
    if jp.name.startswith("_"):
        continue
    try:
        rel = jp.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        continue
    parts = rel.parts
    if len(parts) < 3:
        continue
    state, scope, slug = parts[0], parts[1], parts[2]
    jurisdiction = f"{state}/{scope}/{slug}"
    for text in _harvest_snippets(jp):
        items.append({"jurisdiction": jurisdiction, "source": str(rel), "text": text})
    if len(items) >= EMBED_MAX_ITEMS:
        break

items = items[:EMBED_MAX_ITEMS]
print(f"Embedding {len(items)} snippet(s) from {GEMMA_JSON_ROOT}")

if not items:
    print("Demo 6: no JSON snippets to embed yet — run Demos 1–3 first.")
else:
    if not GEMINI_API_KEY:
        raise RuntimeError("Demo 6 needs GEMINI_API_KEY (AI Studio) for EmbeddingGemma, or skip this cell.")
    vectors = embed_text_with_gemma(
        api_key=GEMINI_API_KEY,
        model=EMBEDDING_MODEL,
        texts=[it["text"] for it in items],
    )
    for it, v in zip(items, vectors):
        it["embedding_dims"] = len(v)

    embeddings_root = PIPE.root / "03_processed_outputs" / "04_embeddings"
    embeddings_root.mkdir(parents=True, exist_ok=True)
    items_path = embeddings_root / "items.jsonl"
    items_path.write_text(
        "\n".join(_json.dumps(it, ensure_ascii=False) for it in items) + "\n",
        encoding="utf-8",
    )

    sim = cosine_similarity_matrix(vectors)
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i]["jurisdiction"] == items[j]["jurisdiction"]:
                continue  # we want CROSS-jurisdiction matches
            if sim[i][j] < EMBED_SIM_THRESHOLD:
                continue
            pairs.append({
                "similarity": round(float(sim[i][j]), 4),
                "left": {"jurisdiction": items[i]["jurisdiction"], "text": items[i]["text"][:200]},
                "right": {"jurisdiction": items[j]["jurisdiction"], "text": items[j]["text"][:200]},
            })
    pairs.sort(key=lambda p: -p["similarity"])
    pairs = pairs[:EMBED_REPORT_TOP_N]

    pairs_path = embeddings_root / "cross_jurisdiction_pairs.json"
    pairs_path.write_text(
        _json.dumps({"threshold": EMBED_SIM_THRESHOLD, "pairs": pairs}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Demo 6 done — {len(pairs)} cross-jurisdiction pair(s) ≥ {EMBED_SIM_THRESHOLD}.")
    for p in pairs[:5]:
        print(f"  {p['similarity']:.3f}  {p['left']['jurisdiction']}  ↔  {p['right']['jurisdiction']}")
        print(f"            L: {p['left']['text'][:90]}")
        print(f"            R: {p['right']['text'][:90]}")
    print(f"  items   → {items_path.relative_to(PIPE.root)}")
    print(f"  pairs   → {pairs_path.relative_to(PIPE.root)}")


## 13) Safety Review — ShieldGemma policy check

Two of our demos generate content that downstream consumers (researchers, journalists, civic-tech tools) may publish or republish:

- **Demo 5** assigns *perceived* demographics to public officials from contact-page photos.
- **Demo 4** processes audio of public comments which can contain hate speech, threats, or harassment.

ShieldGemma runs a fast structured Yes/No check on each generated output for the standard four harm categories (`dangerous_content`, `harassment`, `hate_speech`, `sexually_explicit`). The review writes one verdict file per source so the pipeline has an auditable safety trail before anything leaves the runtime.

Outputs:

- `…/05_safety_review/<mirrored>/<source_stem>.shield.json` — per-source verdicts.
- `…/05_safety_review/_summary.json` — aggregate flag counts.

In [ ]:
# Safety Review — run ShieldGemma over Demo 4 audio summaries and Demo 5 image outputs.
import json as _json

SHIELD_MAX_FILES = int(os.environ.get("GOVERNANCE_SHIELD_MAX_FILES", "40"))

safety_root = PIPE.root / "03_processed_outputs" / "05_safety_review"
safety_root.mkdir(parents=True, exist_ok=True)

reviewed = []
flagged_count = 0
def _try_review(label, source_path, text_for_review):
    global flagged_count
    if not text_for_review or not text_for_review.strip():
        return
    try:
        if not GEMINI_API_KEY:
            print(f"  Skip shield ({label}): set GEMINI_API_KEY for AI Studio ShieldGemma.")
            return
        result = shield_review_text(
            api_key=GEMINI_API_KEY,
            model=SHIELD_MODEL,
            content=text_for_review[:4000],
            user_prompt=f"(automated {label} output from open-navigator pipeline)",
        )
    except Exception as e:
        print(f"  ! shield call failed for {source_path}: {e}")
        return
    rel = source_path.resolve().relative_to(GEMMA_JSON_ROOT.resolve()) if source_path.is_relative_to(GEMMA_JSON_ROOT) else Path(source_path.name)
    out = safety_root / rel.with_suffix(".shield.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(_json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    reviewed.append({"source": str(rel), "flagged": result["flagged"], "categories": result["categories"]})
    if result["flagged"]:
        flagged_count += 1
        print(f"  ⚠ flagged: {rel} — {result['rationale']}")

# Demo 5 outputs — contact-image perception JSONs.
demo5_count = 0
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.image_triage.json")):
    if demo5_count >= SHIELD_MAX_FILES // 2:
        break
    try:
        data = _json.loads(jp.read_text(encoding="utf-8"))
    except Exception:
        continue
    text = _json.dumps(data, ensure_ascii=False)
    _try_review("demo5_image_triage", jp, text)
    demo5_count += 1

# Demo 4 outputs — per-chunk policy analysis JSONs.
demo4_count = 0
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.chunk_*.json")):
    if demo4_count >= SHIELD_MAX_FILES // 2:
        break
    try:
        data = _json.loads(jp.read_text(encoding="utf-8"))
    except Exception:
        continue
    text = _json.dumps(data, ensure_ascii=False)
    _try_review("demo4_audio_chunk", jp, text)
    demo4_count += 1

summary = {
    "model": SHIELD_MODEL,
    "reviewed_count": len(reviewed),
    "flagged_count": flagged_count,
    "reviewed": reviewed,
}
(safety_root / "_summary.json").write_text(
    _json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

if reviewed:
    print(f"\nShieldGemma review done — {len(reviewed)} item(s) reviewed, {flagged_count} flagged.")
    print(f"  summary → {(safety_root / '_summary.json').relative_to(PIPE.root)}")
else:
    print("Safety review: no Demo 4 / Demo 5 outputs found yet — run those demos first.")


## 14) Troubleshooting

- **`Tensor.item() cannot be called on meta tensors` / `processor_kwargs` on Gatekeeper:** restart runtime,
  re-run §2–§3, then §4. Pull latest `gemma_hf_backend.py` (uses `torch_dtype`, `processor_kwargs` for
  `max_soft_tokens`, and proper GPU placement). If it persists on GPU, set
  `GOVERNANCE_LLM_BACKEND=google` + `GEMINI_API_KEY` for Gatekeeper only.
- **`pip` dependency conflicts (torch 2.12 vs torchaudio 2.10, pillow vs gradio):** caused by
  `pip install -U transformers`, which upgrades **torch** while Colab’s **torchaudio** stays pinned.
  **Runtime → Restart session**, then re-run §2 **without** `-U` on transformers (the install cell uses
  plain `pip install -q …`). You can ignore **gradio ↔ pillow** warnings unless you use Gradio in
  this notebook — they do not affect Gatekeeper or the demos.
- **`gemma-4-…` model id not found:** the resolver should auto-fall-back to an available Gemma 4 id. If it raises with a list of visible ids, the E2B/E4B edge variants you're requesting aren't hosted on your project — pick one of the listed ids (typically `gemma-4-26b-a4b-it` or `gemma-4-31b-it`) and set `GOVERNANCE_GENAI_MODEL` / `GOVERNANCE_GATEKEEPER_MODEL` accordingly.
- **`ThinkingConfig` / `MediaResolution` AttributeError:** upgrade
  `google-genai` (`%pip install -q -U "google-genai>=1.0"`). Older builds expose
  the older API surface only; the helper falls back silently when the enum is
  missing but you'll see the model default budget instead of HIGH/LOW.
- **`ffmpeg: command not found` in Demo 4:** Colab ships ffmpeg by default; locally
  install with `apt-get install -y ffmpeg` (Linux) or `brew install ffmpeg` (macOS).
- **`PyMuPDF` import error:** re-run the install cell — `pymupdf` must be present
  for Demo 1 and Demo 2.
- **Payload too big for one request:** reduce `GOVERNANCE_DEMO_MAX_PAGES_PER_PDF`
  or use the Files API (not wired up here) for very long audio.
- **Quota / cost spikes:** lower the demo caps (`MAX_PDFS_PER_JUR`, `MAX_AUDIO_CHUNKS`)
  while you iterate.
- **`/content/governance_pipeline_data/` keeps appearing in the file browser:**
  This is ephemeral Colab runtime disk, NOT Google Drive — files there vanish on disconnect.
  Earlier versions of [`scripts/colab/colab_paths.py`](../colab/colab_paths.py) defaulted to
  `repo.parent / "governance_pipeline_data"` (which became `/content/governance_pipeline_data`
  since the repo clones to `/content/open-navigator`), and `PIPE.ensure_dirs()`
  ([`scripts/utils/gdrive_paths.py`](../utils/gdrive_paths.py) `GovernancePipelinePaths.ensure_dirs`)
  `mkdir -p`'d the empty stage shell. The current bootstrap cell **deletes that folder on every run**
  (`shutil.rmtree(/content/governance_pipeline_data)`) and the resolver now **raises** instead of
  falling back to `/content/` — so the only way data lands there now is if you put it there
  manually. Real data must live on Drive (`MyDrive/CommunityOne/hackathons/2026_Gemma_4_Good/` or
  `MyDrive/CommunityOne/governance_pipeline_data/`) or be pointed at via
  `os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"]` *before* the bootstrap cell.